In [18]:
import os
import numpy as np
from ultralytics import YOLO

# 1. Setup paths and workspace
project_root = r'C:\Users\ndvam\PycharmProjects\model_b'
os.chdir(project_root)

# 2. Define operational thresholds and paths
CONF_THRESHOLD = 0.46
IOU_THRESHOLD = 0.5

# ===================================================================================
# CONFIGURATION & FLAGS (Set perfectly to match SWEEP results)
# ===================================================================================
# True  = נעילה פר-מחלקה (כמו בעקומות P/R הרשמיות)
PER_CLASS_LOCK = True

# False = NMS רגיל מודע-מחלקה (כמו שהוגדר בקובץ הסוויפ המקורי)
AGNOSTIC_NMS = False
# ===================================================================================

images_dir = r'C:\Users\ndvam\PycharmProjects\model_b\datasets\data\valid\images'
labels_dir = r'C:\Users\ndvam\PycharmProjects\model_b\datasets\data\valid\labels'

# 3. Load model and structure classes
model = YOLO('model.pt')
names = model.names
num_classes = len(names)

# Initialize confusion matrix: size (num_classes + 1) to include background row/col
confusion_matrix = np.zeros((num_classes + 1, num_classes + 1), dtype=int)

def calculate_iou(box1, box2):
    """Calculate Intersection over Union (IoU) between two boxes [x1, y1, x2, y2]"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

# 4. Get ground truth files
label_files = sorted([f for f in os.listdir(labels_dir) if f.endswith('.txt')])

print("Processing validation dataset to match SWEEP results...")

# 5. Run prediction box-by-box
results = model.predict(
    source=images_dir,
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    device=0,
    stream=True,
    verbose=False,
    agnostic_nms=AGNOSTIC_NMS,
    imgsz=640,          # <-- חסר בקוד המקורי
    half=False,         # <-- חסר בקוד המקורי
    max_det=1000        # <-- חסר בקוד המקורי
)

for i, result in enumerate(results):
    if i >= len(label_files):
        break

    print(f"Evaluating image {i+1}/{len(label_files)}: {os.path.basename(result.path)}", end='\r')

    # Extract predicted boxes: [x1, y1, x2, y2, conf, class_id]
    if result.boxes is not None and len(result.boxes) > 0:
        pred_boxes = result.boxes.data.cpu().numpy()
        # מיון התחזיות לפי Confidence בסדר יורד - קריטי לדיוק
        pred_boxes = pred_boxes[np.argsort(-pred_boxes[:, 4])]
    else:
        pred_boxes = []

    # Load all ground truth targets for this image
    gt_boxes = []
    label_path = os.path.join(labels_dir, label_files[i])
    if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
        gt_data = np.loadtxt(label_path).reshape(-1, 5)
        h, w = result.orig_shape
        for box in gt_data:
            cls, xc, yc, bw, bh = box
            gt_boxes.append([
                int(cls),
                (xc - bw/2) * w, (yc - bh/2) * h,
                (xc + bw/2) * w, (yc + bh/2) * h
            ])

    matched_gts = set()

    # Match predictions to ground truth targets
    for pred in pred_boxes:
        p_box = pred[0:4]
        p_cls = int(pred[5])

        best_iou = 0
        best_gt_idx = -1

        for gt_idx, gt in enumerate(gt_boxes):
            g_cls = gt[0]

            # מניעת בלבול בין מחלקות: פרדיקציה נבדקת רק מול GT מאותה מחלקה
            if g_cls != p_cls:
                continue

            if PER_CLASS_LOCK:
                if (gt_idx, p_cls) in matched_gts:
                    continue
            else:
                if gt_idx in matched_gts:
                    continue

            g_box = gt[1:5]
            iou = calculate_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_iou >= IOU_THRESHOLD and best_gt_idx != -1:
            g_cls = gt_boxes[best_gt_idx][0]
            confusion_matrix[g_cls, p_cls] += 1

            if PER_CLASS_LOCK:
                matched_gts.add((best_gt_idx, p_cls))
            else:
                matched_gts.add(best_gt_idx)
        else:
            # False Positive
            confusion_matrix[num_classes, p_cls] += 1

    # Track False Negatives
    for gt_idx, gt in enumerate(gt_boxes):
        g_cls = gt[0]
        if PER_CLASS_LOCK:
            if (gt_idx, g_cls) not in matched_gts:
                confusion_matrix[g_cls, num_classes] += 1
        else:
            if gt_idx not in matched_gts:
                confusion_matrix[g_cls, num_classes] += 1

print("\nEvaluation complete!\n")

# 6. Format and display the complete confusion matrix
print("=" * 60)
print(f" CONFUSION MATRIX (At Conf={CONF_THRESHOLD}, IoU={IOU_THRESHOLD}) ")
print(f" Flags: PER_CLASS_LOCK={PER_CLASS_LOCK} | AGNOSTIC_NMS={AGNOSTIC_NMS} ")
print("=" * 60)

header = f"{'True \ Pred':<15}" + "".join([f"{names[idx]:<15}" for idx in range(num_classes)]) + f"{'background':<15}"
print(header)
print("-" * len(header))

for idx in range(num_classes + 1):
    row_label = names[idx] if idx < num_classes else "background"
    row_values = "".join([f"{int(confusion_matrix[idx][jdx]):<15}" for jdx in range(num_classes + 1)])
    print(f"{row_label:<15}{row_values}")

print("=" * 60)

<>:155: SyntaxWarning: invalid escape sequence '\ '
<>:155: SyntaxWarning: invalid escape sequence '\ '
C:\Users\ndvam\AppData\Local\Temp\ipykernel_56476\980583518.py:155: SyntaxWarning: invalid escape sequence '\ '
  header = f"{'True \ Pred':<15}" + "".join([f"{names[idx]:<15}" for idx in range(num_classes)]) + f"{'background':<15}"


Processing validation dataset to match SWEEP results...
Evaluating image 154/154: vid3_highres_MP4-0153_jpg.rf.5ed9df886d635ed9ed74d8ec164d82ea.jpg
Evaluation complete!

 CONFUSION MATRIX (At Conf=0.46, IoU=0.5) 
 Flags: PER_CLASS_LOCK=True | AGNOSTIC_NMS=False 
True \ Pred    fish           partial fish   background     
------------------------------------------------------------
fish           243            0              75             
partial fish   0              552            178            
background     26             114            0              
